In [1]:
#%% [CELL 1] ENVIRONMENT SETUP (KAGGLE SAFE)
# =============================================================================
# Goal: Avoid source builds for fragile packages (onnx, pycocotools).
# Strategy:
# 1) Pin wheel-friendly versions in constraints.
# 2) Install those packages from binary wheels first.
# 3) Install super-gradients without dependency re-resolution.
# =============================================================================

import os
import sys
import subprocess


def run_command(command, message, strict=True):
    print(f"\n--> {message}")
    print(command)
    try:
        subprocess.check_call(command, shell=True)
    except subprocess.CalledProcessError as e:
        if strict:
            raise
        print(f"WARNING: Command failed ({e.returncode}): {command}")


# 1. Constraints file (global resolver guardrails)
constraints = """numpy<2.0
pycocotools==2.0.7
onnx==1.16.1
"""
with open("constraints.txt", "w") as f:
    f.write(constraints)
print("--> 1. Created constraints.txt")

# 2. Optional system deps (best effort on Kaggle)
run_command(
    "apt-get update && apt-get install -y libgl1-mesa-glx libglib2.0-0",
    "Installing system runtime libraries...",
    strict=False,
)

# 3. Clean conflicting installs
pkgs = "super-gradients pycocotools onnx"
run_command(
    f"{sys.executable} -m pip uninstall -y {pkgs}",
    "Removing potentially conflicting packages...",
    strict=False,
)

# 4. Base packaging tools + numpy pin
run_command(
    f"{sys.executable} -m pip install -U pip setuptools wheel 'numpy<2.0' -c constraints.txt",
    "Installing packaging tools and pinned numpy...",
)

# 5. Install fragile deps from wheels only (no source builds)
run_command(
    f"{sys.executable} -m pip install --only-binary=:all: --prefer-binary pycocotools onnx -c constraints.txt",
    "Installing pycocotools/onnx from binary wheels...",
)

# 6. Install super-gradients and force binary wheels for fragile deps
run_command(
    f"{sys.executable} -m pip install super-gradients --prefer-binary --only-binary=onnx,pycocotools -c constraints.txt",
    "Installing super-gradients with binary-only onnx/pycocotools...",
)

# 7. Install remaining project libs
run_command(
    f"{sys.executable} -m pip install -U ultralytics roboflow -c constraints.txt",
    "Installing Ultralytics and Roboflow...",
)

# 8. Verification
import numpy as np
import torch
import onnx
import pycocotools
import super_gradients

print(f"\n{'='*20} VERIFICATION {'='*20}")
print(f"Python: {sys.version.split()[0]}")
print(f"Numpy: {np.__version__}")
print(f"ONNX: {onnx.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Super-Gradients import: OK ({super_gradients.__version__ if hasattr(super_gradients, '__version__') else 'version not exposed'})")

--> 1. Created constraints.txt

--> Installing system runtime libraries...
apt-get update && apt-get install -y libgl1-mesa-glx libglib2.0-0
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.0 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,361 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)



Building dependency tree...
Reading state information...
libgl1-mesa-glx is already the newest version (23.0.4-0ubuntu1~22.04.1).
The following additional packages will be installed:
  libglib2.0-bin libglib2.0-dev libglib2.0-dev-bin
Suggested packages:
  libglib2.0-doc libxml2-utils
Recommended packages:
  xdg-user-dirs
The following packages will be upgraded:
  libglib2.0-0 libglib2.0-bin libglib2.0-dev libglib2.0-dev-bin
4 upgraded, 0 newly installed, 0 to remove and 168 not upgraded.
Need to get 3,408 kB of archives.
After this operation, 10.2 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libglib2.0-dev amd64 2.72.4-0ubuntu2.9 [1,744 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libglib2.0-dev-bin amd64 2.72.4-0ubuntu2.9 [117 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libglib2.0-bin amd64 2.72.4-0ubuntu2.9 [80.9 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates/main am

Found existing installation: pycocotools 2.0.10
Uninstalling pycocotools-2.0.10:
  Successfully uninstalled pycocotools-2.0.10
Found existing installation: onnx 1.20.1
Uninstalling onnx-1.20.1:
  Successfully uninstalled onnx-1.20.1

--> Installing packaging tools and pinned numpy...
/usr/bin/python3 -m pip install -U pip setuptools wheel 'numpy<2.0' -c constraints.txt
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 85.8 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.45.1
    Uninstalling wheel-0.45.1:
      Successfully uninstalled wheel-0.45.1
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires google-auth==2.38.0, but you have google-auth 2.47.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2


--> Installing pycocotools/onnx from binary wheels...
/usr/bin/python3 -m pip install --only-binary=:all: --prefer-binary pycocotools onnx -c constraints.txt

The conflict is caused by:
    The user requested pycocotools
    The user requested (constraint) pycocotools==2.0.7

Additionally, some packages in these conflicts have no matching distributions available for your environment:
    pycocotools

To fix this you could try to:
1. loosen the range of package versions you've specified
2. remove package versions to allow pip to attempt to solve the dependency conflict



ERROR: Cannot install pycocotools because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


CalledProcessError: Command '/usr/bin/python3 -m pip install --only-binary=:all: --prefer-binary pycocotools onnx -c constraints.txt' returned non-zero exit status 1.

In [3]:
import sys
sys.version

'3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]'

In [ ]:
import os
#from roboflow import Roboflow
import numpy as np
from ultralytics import YOLO
import sys
import time
import yaml
import torch
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import glob
import cv2

# Check for GPU availability - Critical for training and latency tests
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"--> Hardware Check: Using {DEVICE} ({torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'})")

if DEVICE == 'cpu':
    print("WARNING: No GPU detected. Training will be extremely slow.")

# Constants
PROJECT_NAME = "Pothole_Research"
IMG_SIZE = 640
BATCH_SIZE = 16
EPOCHS = 50  # Set to 50-100 for actual research results

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("ROBOFLOW_API_KEY")

In [ ]:
def setup_dataset(api_key):
    print("\n--> Downloading Dataset from Roboflow...")
    rf = Roboflow(api_key=api_key)
    project = rf.workspace("smartathon").project("new-pothole-detection")
    # v8 format is compatible with v8, v9, v11, v12
    dataset = project.version(2).download("yolov8") 
    
    # Fix paths in data.yaml
    yaml_path = os.path.join(dataset.location, "data.yaml")
    
    with open(yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)
    
    # Force absolute paths
    data_config['train'] = os.path.join(os.getcwd(), dataset.location, "train", "images")
    data_config['val'] = os.path.join(os.getcwd(), dataset.location, "valid", "images")
    data_config['test'] = os.path.join(os.getcwd(), dataset.location, "test", "images")
    
    # Overwrite the yaml
    with open(yaml_path, 'w') as f:
        yaml.dump(data_config, f)
        
    print(f"--> Dataset Configured at: {yaml_path}")
    return yaml_path, dataset.location

In [ ]:
# Execution
try:
    DATA_YAML, DATASET_ROOT = setup_dataset(ROBOFLOW_API_KEY)
except Exception as e:
    print(f"Error downloading data. Check API Key. Details: {e}")
    # Fallback for testing if data already exists locally
    DATA_YAML = "New-Pothole-Detection-2/data.yaml" 
    DATASET_ROOT = "New-Pothole-Detection-2"

In [ ]:
# EDA (EXPLORATORY DATA ANALYSIS)
# =============================================================================
# Expanded dataset diagnostics:
# 1) Split-level health checks (images/labels/annotations/missing labels)
# 2) Bounding-box geometry distributions and percentile report
# 3) Spatial heatmap of box centers to reveal annotation bias
# 4) Sample ground-truth overlays for quick qualitative inspection
# =============================================================================

def perform_eda(dataset_root, num_samples=6):
    print(f"\n{'='*10} STARTING EDA {'='*10}")

    split_aliases = {
        "train": ["train"],
        "valid": ["valid", "val"],
        "test": ["test"]
    }

    def collect_split_files(root, split_candidates):
        image_files, label_files = [], []
        for split_name in split_candidates:
            img_dir = os.path.join(root, split_name, "images")
            lbl_dir = os.path.join(root, split_name, "labels")

            if os.path.isdir(img_dir):
                image_files.extend(sorted(glob.glob(os.path.join(img_dir, "*.jpg"))))
                image_files.extend(sorted(glob.glob(os.path.join(img_dir, "*.jpeg"))))
                image_files.extend(sorted(glob.glob(os.path.join(img_dir, "*.png"))))

            if os.path.isdir(lbl_dir):
                label_files.extend(sorted(glob.glob(os.path.join(lbl_dir, "*.txt"))))

        return sorted(image_files), sorted(label_files)

    split_summary = []
    all_box_records = []

    for split_key, aliases in split_aliases.items():
        images, labels = collect_split_files(dataset_root, aliases)
        label_lookup = {
            Path(lbl).stem: lbl
            for lbl in labels
        }

        annotated_images = 0
        missing_label_images = 0
        empty_label_files = 0
        ann_count = 0

        for img_path in images:
            stem = Path(img_path).stem
            label_path = label_lookup.get(stem)

            if not label_path or not os.path.exists(label_path):
                missing_label_images += 1
                continue

            with open(label_path, 'r') as f:
                lines = [ln.strip() for ln in f.readlines() if ln.strip()]

            if not lines:
                empty_label_files += 1
                continue

            annotated_images += 1
            ann_count += len(lines)

            for line in lines:
                parts = line.split()
                if len(parts) != 5:
                    continue

                cls, x, y, w, h = map(float, parts)
                area = w * h
                aspect_ratio = (w / h) if h > 0 else np.nan

                all_box_records.append({
                    "split": split_key,
                    "class_id": int(cls),
                    "x_center": x,
                    "y_center": y,
                    "width": w,
                    "height": h,
                    "area": area,
                    "aspect_ratio": aspect_ratio
                })

        split_summary.append({
            "Split": split_key,
            "Images": len(images),
            "Label files": len(labels),
            "Annotated images": annotated_images,
            "Missing labels": missing_label_images,
            "Empty label files": empty_label_files,
            "Annotations": ann_count,
            "Avg boxes / image": (ann_count / len(images)) if images else 0
        })

    split_df = pd.DataFrame(split_summary)
    print("\n--- DATASET HEALTH REPORT ---")
    print(split_df.to_string(index=False))

    if not all_box_records:
        print("\nNo valid annotations found; skipping visualization steps.")
        return

    box_df = pd.DataFrame(all_box_records)

    print("\n--- BOUNDING BOX STATISTICS REPORT ---")
    stats_report = box_df[["width", "height", "area", "aspect_ratio"]].describe(
        percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95]
    ).T
    print(stats_report)

    small_threshold = 0.01
    small_count = int((box_df["area"] < small_threshold).sum())
    print(f"\nTotal annotations: {len(box_df)}")
    print(f"Small potholes (<1% normalized area): {small_count} / {len(box_df)} ({(small_count / len(box_df) * 100):.2f}%)")

    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(2, 3, figsize=(20, 11))

    sns.histplot(box_df["area"], bins=60, kde=True, color='steelblue', ax=axes[0, 0])
    axes[0, 0].axvline(small_threshold, color='crimson', linestyle='--', label='1% small-object threshold')
    axes[0, 0].set_title("Box Area Distribution")
    axes[0, 0].set_xlabel("Normalized Area")
    axes[0, 0].legend()

    sns.histplot(box_df["width"], bins=50, color='teal', ax=axes[0, 1])
    axes[0, 1].set_title("Width Distribution")
    axes[0, 1].set_xlabel("Normalized Width")

    sns.histplot(box_df["height"], bins=50, color='darkorange', ax=axes[0, 2])
    axes[0, 2].set_title("Height Distribution")
    axes[0, 2].set_xlabel("Normalized Height")

    sns.boxplot(data=box_df, x="split", y="area", palette='Set2', ax=axes[1, 0])
    axes[1, 0].set_title("Area by Split")
    axes[1, 0].set_xlabel("Split")
    axes[1, 0].set_ylabel("Normalized Area")

    sns.scatterplot(
        data=box_df,
        x="area",
        y="aspect_ratio",
        hue="split",
        s=14,
        alpha=0.35,
        ax=axes[1, 1]
    )
    axes[1, 1].set_title("Area vs Aspect Ratio")
    axes[1, 1].set_xlabel("Normalized Area")
    axes[1, 1].set_ylabel("Aspect Ratio (W/H)")
    finite_aspect = box_df["aspect_ratio"].replace([np.inf, -np.inf], np.nan).dropna()
    aspect_ymax = np.percentile(finite_aspect, 99) * 1.1 if len(finite_aspect) else 5
    axes[1, 1].set_ylim(0, max(aspect_ymax, 1))

    sns.kdeplot(
        data=box_df,
        x="x_center",
        y="y_center",
        fill=True,
        cmap="rocket",
        levels=20,
        thresh=0.02,
        ax=axes[1, 2]
    )
    axes[1, 2].invert_yaxis()  # Image coordinates: origin near top-left
    axes[1, 2].set_title("Annotation Center Density")
    axes[1, 2].set_xlabel("X center (normalized)")
    axes[1, 2].set_ylabel("Y center (normalized)")

    plt.tight_layout()
    plt.show()

    # CDF highlights what fraction of objects are below a given scale.
    sorted_area = np.sort(box_df["area"].values)
    cdf = np.arange(1, len(sorted_area) + 1) / len(sorted_area)

    plt.figure(figsize=(8, 5))
    plt.plot(sorted_area, cdf, color='purple', linewidth=2)
    plt.axvline(small_threshold, color='crimson', linestyle='--', label='1% threshold')
    plt.title("Cumulative Distribution of Box Area")
    plt.xlabel("Normalized Area")
    plt.ylabel("Cumulative Fraction")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Sample images with GT annotations (train split preferred)
    train_candidates = split_aliases["train"]
    train_images, _ = collect_split_files(dataset_root, train_candidates)

    if not train_images:
        print("No train images found for qualitative preview.")
        return

    n_show = min(num_samples, len(train_images))
    sample_indices = np.linspace(0, len(train_images) - 1, n_show, dtype=int)

    print("\nVisualizing Ground Truth Samples...")
    plt.figure(figsize=(4 * n_show, 4))

    for i, idx in enumerate(sample_indices):
        img_path = train_images[idx]
        label_path = img_path.replace("images", "labels")
        label_path = os.path.splitext(label_path)[0] + ".txt"

        img = cv2.imread(img_path)
        if img is None:
            continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h_img, w_img, _ = img.shape

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue

                    _, x, y, w, h = map(float, parts)
                    x1 = int((x - w / 2) * w_img)
                    y1 = int((y - h / 2) * h_img)
                    x2 = int((x + w / 2) * w_img)
                    y2 = int((y + h / 2) * h_img)
                    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

        plt.subplot(1, n_show, i + 1)
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"Sample {i + 1}")

    plt.tight_layout()
    plt.show()


try:
    perform_eda(DATASET_ROOT)
except Exception as e:
    print(f"EDA Skipped due to error: {e}")



In [ ]:
# CUSTOM ARCHITECTURE: YOLO-FPN
# =============================================================================
# Here we define a custom YOLOv8 variant.
# We replace the heavy PANet (Path Aggregation Network) with a standard FPN.
# Hypothesis: Removing the bottom-up path reduces FLOPs/Latency for drone chips.
# =============================================================================

yolo_fpn_config = """
# Ultralytics YOLOv8-FPN Custom Configuration
nc: 1  # number of classes
scales: # model compound scaling constants, i.e. 'n'
  n: [0.33, 0.25, 1024]

backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv, [64, 3, 2]]  # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]  # 1-P2/4
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]  # 3-P3/8
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]  # 5-P4/16
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]  # 7-P5/32
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]  # 9

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]  # cat backbone P4
  - [-1, 3, C2f, [512]]  # 12

  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]]  # cat backbone P3
  - [-1, 3, C2f, [256]]  # 15 (P3/8-small)

  # FPN Head (Standard Detect, no further bottom-up aggregation)
  - [[15, 12, 9], 1, Detect, [nc]]  # Detect(P3, P4, P5)
"""

with open("yolov8n-fpn.yaml", "w") as f:
    f.write(yolo_fpn_config)
print("--> Custom YOLO-FPN Architecture Saved.")

In [ ]:
# TRAINING LOOP (ULTRALYTICS MODELS)
# =============================================================================
# Trains v8, v9, v12, and the custom FPN model using the Ultralytics API.
# Supports pause/resume using per-run checkpoints (weights/last.pt).
# Re-run this cell after interruption to continue training from where it stopped.
# =============================================================================

from ultralytics import YOLO

# Optional controls
FORCE_RETRAIN = False  # True = ignore checkpoints and start each model from scratch

# Dictionary mapping display names to model sources
models_config = {
    "YOLOv8n": "yolov8n.pt",       # Baseline
    "YOLOv9c": "yolov9c.pt",       # PGI/GELAN (Deep layers)
    "YOLOv11": "yolo11n.pt",
    "YOLOv12n": "yolo12n.pt",      # Attention Mechanism (requires v12 installed)
    "YOLOv8-FPN": "yolov8n-fpn.yaml" # Custom Config (scratch training)
}

benchmark_results = {}

def train_ultralytics_model(model_name, model_src):
    run_dir = Path(PROJECT_NAME) / model_name
    last_ckpt = run_dir / "weights" / "last.pt"
    best_ckpt = run_dir / "weights" / "best.pt"
    results_csv = run_dir / "results.csv"

    fresh_train_args = {
        "data": DATA_YAML,
        "epochs": EPOCHS,
        "imgsz": IMG_SIZE,
        "batch": BATCH_SIZE,
        "project": PROJECT_NAME,
        "name": model_name,
        "exist_ok": True,
        "verbose": False,
        "device": 0 if DEVICE == 'cuda' else 'cpu'
    }

    run_complete = False
    if results_csv.exists() and not FORCE_RETRAIN:
        try:
            history = pd.read_csv(results_csv)
            # Ultralytics writes one row per finished epoch.
            run_complete = len(history) >= EPOCHS
        except Exception:
            run_complete = False

    if run_complete:
        print(f"{model_name} already completed {EPOCHS} epochs. Skipping train and using saved weights.")
    elif last_ckpt.exists() and not FORCE_RETRAIN:
        print(f"Resuming {model_name} from checkpoint: {last_ckpt}")
        model = YOLO(str(last_ckpt))
        model.train(resume=True)
    else:
        if FORCE_RETRAIN and run_dir.exists():
            print(f"FORCE_RETRAIN=True -> starting {model_name} from source weights/config.")
        model = YOLO(model_src)
        model.train(**fresh_train_args)

    eval_weights = best_ckpt if best_ckpt.exists() else (last_ckpt if last_ckpt.exists() else model_src)
    eval_model = YOLO(str(eval_weights))

    metrics = eval_model.val(
        data=DATA_YAML,
        split='val',
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=0 if DEVICE == 'cuda' else 'cpu',
        verbose=False
    )

    params_m = 0
    try:
        info_output = eval_model.info(verbose=False)
        if isinstance(info_output, (list, tuple)) and len(info_output) > 1:
            params_m = info_output[1]
    except Exception:
        pass

    # Export the trained checkpoint for deployment simulation.
    try:
        eval_model.export(format='onnx', dynamic=True)
    except Exception as export_err:
        print(f"ONNX export failed for {model_name}: {export_err}")

    return {
        "mAP50": metrics.box.map50,
        "mAP50-95": metrics.box.map,
        "Parameters": params_m
    }

try:
    for model_name, model_src in models_config.items():
        print(f"\n{'='*10} Training {model_name} {'='*10}")

        try:
            benchmark_results[model_name] = train_ultralytics_model(model_name, model_src)
        except Exception as e:
            print(f"FAILED to train {model_name}: {e}")
            benchmark_results[model_name] = {"mAP50": 0, "mAP50-95": 0, "Parameters": 0}
except KeyboardInterrupt:
    print("\nTraining interrupted manually. Re-run this cell to resume unfinished models from last.pt checkpoints.")
    raise


In [ ]:
# TRAINING LOOP (YOLO-NAS)
# =============================================================================
# YOLO-NAS uses the SuperGradients library (Deci AI).
# It requires a different data loader setup (COCO format conversion or direct loading).
# =============================================================================

try:
    from super_gradients.training import Trainer, models
    from super_gradients.training.dataloaders.dataloaders import coco_detection_yolo_format_train, coco_detection_yolo_format_val
    from super_gradients.training.losses import PPYoloELoss
    from super_gradients.training.metrics import DetectionMetrics_050
    from super_gradients.training.models.detection_models.pp_yolo_e import PPYoloEPostPredictionCallback

    print(f"\n{'='*10} Training YOLO-NAS-S {'='*10}")
    
    # Initialize Trainer
    trainer = Trainer(experiment_name="yolo_nas_pothole", ckpt_root_dir=f"{PROJECT_NAME}/YOLO-NAS")

    # Define Data Loaders
    train_data = coco_detection_yolo_format_train(
        dataset_params={
            "data_dir": DATASET_ROOT,
            "images_dir": "train/images",
            "labels_dir": "train/labels",
            "classes": ["pothole"]
        },
        dataloader_params={"batch_size": BATCH_SIZE, "num_workers": 2}
    )

    val_data = coco_detection_yolo_format_val(
        dataset_params={
            "data_dir": DATASET_ROOT,
            "images_dir": "valid/images",
            "labels_dir": "valid/labels",
            "classes": ["pothole"]
        },
        dataloader_params={"batch_size": BATCH_SIZE, "num_workers": 2}
    )

    # Load Model
    nas_model = models.get("yolo_nas_s", num_classes=1, pretrained_weights="coco")

    # Train Params
    train_params = {
        "max_epochs": EPOCHS,
        "lr_mode": "cosine",
        "initial_lr": 0.001,
        "loss": PPYoloELoss(use_static_assigner=False, num_classes=1, reg_max=16),
        "optimizer": "AdamW",
        "average_best_models": True,
        "metric_to_watch": 'mAP@0.50',
        "greater_metric_to_watch_is_better": True,
        "train_metrics_list": [DetectionMetrics_050(score_thres=0.1, top_k_predictions=300, num_cls=1, normalize_targets=True, post_prediction_callback=PPYoloEPostPredictionCallback(score_threshold=0.01, nms_top_k=1000, max_predictions=300, nms_threshold=0.7))],
        "valid_metrics_list": [DetectionMetrics_050(score_thres=0.1, top_k_predictions=300, num_cls=1, normalize_targets=True, post_prediction_callback=PPYoloEPostPredictionCallback(score_threshold=0.01, nms_top_k=1000, max_predictions=300, nms_threshold=0.7))],
    }

    trainer.train(model=nas_model, training_params=train_params, train_loader=train_data, valid_loader=val_data)
    
    # ACTUAL EXTRACTION: Run validation on best checkpoint to get accurate metrics
    print("--> Extracting Final NAS Metrics...")
    test_metrics = trainer.test(model=nas_model, test_loader=val_data, test_metrics_list=train_params["valid_metrics_list"])
    
    # Calculate Parameters
    total_params = sum(p.numel() for p in nas_model.parameters() if p.requires_grad) / 1e6
    
    # Note: Structure of test_metrics depends on SG version, usually dictionary
    # Typical keys: 'mAP@0.50', 'mAP@0.50:0.95'
    benchmark_results["YOLO-NAS-S"] = {
        "mAP50": test_metrics.get('mAP@0.50', 0.0), 
        "mAP50-95": test_metrics.get('mAP@0.50:0.95', 0.0), 
        "Parameters": total_params
    }

except ImportError:
    print("SuperGradients not installed. Skipping YOLO-NAS.")
except Exception as e:
    print(f"YOLO-NAS Training/Extraction Failed: {e}")
    # Fallback to prevent crash in plotting
    benchmark_results["YOLO-NAS-S"] = {"mAP50": 0, "mAP50-95": 0, "Parameters": 0}

In [ ]:
# MODEL EXPORTS & ARTIFACT LINKS (PT / ONNX / YAML)
# =============================================================================
# Exports each Ultralytics model to ONNX (if needed) and displays FileLinks for:
# - .pt checkpoints
# - .onnx exports
# - key .yaml files (model config / args / dataset yaml)
# =============================================================================

from IPython.display import display, Markdown, FileLink


def export_and_link_model_artifacts(model_map, project_name, data_yaml, force_onnx_export=False):
    print(f"\n{'='*10} EXPORTING MODELS & LINKING ARTIFACTS {'='*10}")

    rows = []

    for model_name, model_src in model_map.items():
        run_dir = Path(project_name) / model_name
        weights_dir = run_dir / "weights"
        best_pt = weights_dir / "best.pt"
        last_pt = weights_dir / "last.pt"

        source_candidates = [best_pt, last_pt, Path(model_src)]
        load_source = next((p for p in source_candidates if p.exists()), None)

        if load_source is None:
            print(f"Skipping {model_name}: no loadable source found (best.pt/last.pt/model_src).")
            continue

        onnx_path = weights_dir / "best.onnx"
        if not onnx_path.exists() or force_onnx_export:
            try:
                model = YOLO(str(load_source))
                model.export(format='onnx', dynamic=True)
                # Ultralytics may place ONNX beside source with same stem.
                exported_candidate = load_source.with_suffix('.onnx')
                if exported_candidate.exists() and exported_candidate != onnx_path:
                    onnx_path = exported_candidate
            except Exception as e:
                print(f"ONNX export failed for {model_name}: {e}")

        pt_files = []
        if best_pt.exists():
            pt_files.append(best_pt)
        if last_pt.exists():
            pt_files.append(last_pt)

        onnx_files = []
        if onnx_path.exists():
            onnx_files.append(onnx_path)
        else:
            discovered_onnx = sorted(run_dir.rglob('*.onnx'))
            onnx_files.extend(discovered_onnx)

        yaml_candidates = []
        model_src_path = Path(model_src)
        if model_src_path.suffix.lower() in ['.yaml', '.yml'] and model_src_path.exists():
            yaml_candidates.append(model_src_path)

        args_yaml = run_dir / 'args.yaml'
        if args_yaml.exists():
            yaml_candidates.append(args_yaml)

        data_yaml_path = Path(data_yaml)
        if data_yaml_path.exists():
            yaml_candidates.append(data_yaml_path)

        # Include any run-level yaml metadata if present.
        yaml_candidates.extend(sorted(run_dir.glob('*.yaml')))
        yaml_candidates.extend(sorted(run_dir.glob('*.yml')))

        seen = set()
        key_yaml_files = []
        for yp in yaml_candidates:
            yp_resolved = str(yp)
            if yp_resolved not in seen:
                key_yaml_files.append(yp)
                seen.add(yp_resolved)

        rows.append({
            'Model': model_name,
            'PT_Files': [str(p) for p in pt_files],
            'ONNX_Files': [str(p) for p in onnx_files],
            'YAML_Files': [str(p) for p in key_yaml_files]
        })

    artifacts_df = pd.DataFrame(rows)

    if artifacts_df.empty:
        print('No model artifacts found.')
        return None

    export_csv = Path(project_name) / 'reports' / 'model_artifact_index.csv'
    export_csv.parent.mkdir(parents=True, exist_ok=True)
    artifacts_df.to_csv(export_csv, index=False)
    print(f"Artifact index saved to: {export_csv}")

    for _, row in artifacts_df.iterrows():
        display(Markdown(f"### {row['Model']}"))

        display(Markdown("**PT files**"))
        if row['PT_Files']:
            for fp in row['PT_Files']:
                display(FileLink(fp))
        else:
            display(Markdown("_No PT files found._"))

        display(Markdown("**ONNX files**"))
        if row['ONNX_Files']:
            for fp in row['ONNX_Files']:
                display(FileLink(fp))
        else:
            display(Markdown("_No ONNX files found._"))

        display(Markdown("**Key YAML files**"))
        if row['YAML_Files']:
            for fp in row['YAML_Files']:
                display(FileLink(fp))
        else:
            display(Markdown("_No YAML files found._"))

    return artifacts_df


try:
    model_artifacts_df = export_and_link_model_artifacts(
        model_map=models_config,
        project_name=PROJECT_NAME,
        data_yaml=DATA_YAML,
        force_onnx_export=False
    )
except Exception as e:
    print(f"Model export/link step failed: {e}")


In [ ]:
# QUALITATIVE MODEL TESTING
# =============================================================================
# Evaluates visual and detection quality on test images.
# Outputs:
# 1) Per-image/per-model metrics (TP/FP/FN, precision/recall/F1)
# 2) Aggregated model report across confidence thresholds
# 3) Comparative plots + qualitative prediction panels
# =============================================================================

def yolo_label_to_xyxy(x, y, w, h, img_w, img_h):
    x1 = max(0, int((x - w / 2) * img_w))
    y1 = max(0, int((y - h / 2) * img_h))
    x2 = min(img_w - 1, int((x + w / 2) * img_w))
    y2 = min(img_h - 1, int((y + h / 2) * img_h))
    return [x1, y1, x2, y2]


def box_iou_xyxy(box_a, box_b):
    xa1, ya1, xa2, ya2 = box_a
    xb1, yb1, xb2, yb2 = box_b

    inter_x1 = max(xa1, xb1)
    inter_y1 = max(ya1, yb1)
    inter_x2 = min(xa2, xb2)
    inter_y2 = min(ya2, yb2)

    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = max(0, xa2 - xa1) * max(0, ya2 - ya1)
    area_b = max(0, xb2 - xb1) * max(0, yb2 - yb1)
    union = area_a + area_b - inter_area

    return inter_area / union if union > 0 else 0.0


def greedy_match(pred_boxes, gt_boxes, iou_thr=0.5):
    if len(pred_boxes) == 0 and len(gt_boxes) == 0:
        return 0, 0, 0
    if len(pred_boxes) == 0:
        return 0, 0, len(gt_boxes)
    if len(gt_boxes) == 0:
        return 0, len(pred_boxes), 0

    matches = []
    for p_idx, p_box in enumerate(pred_boxes):
        for g_idx, g_box in enumerate(gt_boxes):
            iou = box_iou_xyxy(p_box, g_box)
            if iou >= iou_thr:
                matches.append((iou, p_idx, g_idx))

    matches.sort(reverse=True, key=lambda x: x[0])
    used_p = set()
    used_g = set()
    tp = 0

    for _, p_idx, g_idx in matches:
        if p_idx in used_p or g_idx in used_g:
            continue
        used_p.add(p_idx)
        used_g.add(g_idx)
        tp += 1

    fp = len(pred_boxes) - tp
    fn = len(gt_boxes) - tp
    return tp, fp, fn


def load_ground_truth_xyxy(label_path, img_w, img_h):
    gt_boxes = []
    if not os.path.exists(label_path):
        return gt_boxes

    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            _, x, y, w, h = map(float, parts)
            gt_boxes.append(yolo_label_to_xyxy(x, y, w, h, img_w, img_h))
    return gt_boxes


def test_models_qualitative(dataset_root, model_names=None, num_samples=40, conf_thresholds=(0.25, 0.5), iou_match_threshold=0.5):
    print(f"\n{'='*10} QUALITATIVE MODEL TESTING {'='*10}")

    test_images = sorted(glob.glob(os.path.join(dataset_root, "test", "images", "*.jpg")))
    test_images += sorted(glob.glob(os.path.join(dataset_root, "test", "images", "*.jpeg")))
    test_images += sorted(glob.glob(os.path.join(dataset_root, "test", "images", "*.png")))

    if not test_images:
        print("No test images found.")
        return None, None

    if model_names is None:
        model_names = list(models_config.keys())

    reports_dir = Path(PROJECT_NAME) / "reports"
    figs_dir = Path(PROJECT_NAME) / "figures"
    reports_dir.mkdir(parents=True, exist_ok=True)
    figs_dir.mkdir(parents=True, exist_ok=True)

    n_eval = min(num_samples, len(test_images))
    sample_idx = np.linspace(0, len(test_images) - 1, n_eval, dtype=int)
    eval_images = [test_images[i] for i in sample_idx]

    model_objects = {}
    for model_name in model_names:
        trained_weights = Path(PROJECT_NAME) / model_name / "weights" / "best.pt"
        weights_path = str(trained_weights) if trained_weights.exists() else models_config.get(model_name, "yolov8n.pt")
        model_objects[model_name] = YOLO(weights_path)
        print(f"Loaded {model_name} from: {weights_path}")

    per_image_records = []

    for image_path in eval_images:
        img = cv2.imread(image_path)
        if img is None:
            continue
        h_img, w_img = img.shape[:2]
        label_path = image_path.replace("images", "labels")
        label_path = os.path.splitext(label_path)[0] + ".txt"
        gt_boxes = load_ground_truth_xyxy(label_path, w_img, h_img)

        for conf_thr in conf_thresholds:
            for model_name, model in model_objects.items():
                results = model.predict(image_path, conf=float(conf_thr), verbose=False)
                pred = results[0]

                pred_boxes = []
                if pred.boxes is not None and len(pred.boxes) > 0:
                    pred_boxes = pred.boxes.xyxy.detach().cpu().numpy().tolist()

                tp, fp, fn = greedy_match(pred_boxes, gt_boxes, iou_thr=iou_match_threshold)
                precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
                recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
                f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

                per_image_records.append({
                    "Image": os.path.basename(image_path),
                    "Model": model_name,
                    "Conf": float(conf_thr),
                    "GT_Count": len(gt_boxes),
                    "Pred_Count": len(pred_boxes),
                    "TP": tp,
                    "FP": fp,
                    "FN": fn,
                    "Precision": precision,
                    "Recall": recall,
                    "F1": f1
                })

    per_image_df = pd.DataFrame(per_image_records)
    if per_image_df.empty:
        print("No qualitative records generated.")
        return None, None

    summary_df = (
        per_image_df
        .groupby(["Model", "Conf"], as_index=False)
        .agg(
            Images=("Image", "nunique"),
            GT_Objects=("GT_Count", "sum"),
            Pred_Objects=("Pred_Count", "sum"),
            TP=("TP", "sum"),
            FP=("FP", "sum"),
            FN=("FN", "sum"),
            Precision=("Precision", "mean"),
            Recall=("Recall", "mean"),
            F1=("F1", "mean")
        )
    )

    per_image_csv = reports_dir / "qualitative_per_image_metrics.csv"
    summary_csv = reports_dir / "qualitative_model_summary.csv"
    per_image_df.to_csv(per_image_csv, index=False)
    summary_df.to_csv(summary_csv, index=False)

    print("\n--- QUALITATIVE MODEL SUMMARY ---")
    print(summary_df.sort_values(["Conf", "F1"], ascending=[True, False]).to_string(index=False))
    print(f"Saved per-image metrics to: {per_image_csv}")
    print(f"Saved summary report to: {summary_csv}")

    plt.style.use('seaborn-v0_8-whitegrid')

    plt.figure(figsize=(11, 6))
    sns.barplot(data=summary_df, x="Model", y="F1", hue="Conf", palette="Set2")
    plt.title("Qualitative Performance by Model and Confidence")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.savefig(figs_dir / "qualitative_f1_by_model_conf.png", dpi=200)
    plt.show()

    heatmap_df = summary_df.pivot(index="Model", columns="Conf", values="F1")
    plt.figure(figsize=(8, 5))
    sns.heatmap(heatmap_df, annot=True, fmt=".3f", cmap="YlGnBu")
    plt.title("F1 Heatmap (Model vs Confidence)")
    plt.tight_layout()
    plt.savefig(figs_dir / "qualitative_f1_heatmap.png", dpi=200)
    plt.show()

    plt.figure(figsize=(12, 6))
    sns.boxplot(data=per_image_df, x="Model", y="F1", hue="Conf", palette="Set3")
    plt.title("Per-Image F1 Distribution")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.savefig(figs_dir / "qualitative_f1_distribution.png", dpi=200)
    plt.show()

    # Visual panel from best model at primary confidence threshold.
    base_conf = float(conf_thresholds[0])
    best_model_row = summary_df[summary_df["Conf"] == base_conf].sort_values("F1", ascending=False).head(1)
    best_model = best_model_row["Model"].iloc[0] if not best_model_row.empty else summary_df.sort_values("F1", ascending=False)["Model"].iloc[0]

    preview_images = eval_images[:min(4, len(eval_images))]
    preview_model = model_objects[best_model]
    plt.figure(figsize=(16, 4 * len(preview_images)))

    for i, img_path in enumerate(preview_images):
        res = preview_model.predict(img_path, conf=base_conf, verbose=False)[0]
        vis = cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)
        plt.subplot(len(preview_images), 1, i + 1)
        plt.imshow(vis)
        plt.axis('off')
        plt.title(f"{best_model} @ conf={base_conf:.2f} | {os.path.basename(img_path)}")

    plt.tight_layout()
    plt.savefig(figs_dir / "qualitative_best_model_preview.png", dpi=200)
    plt.show()

    return per_image_df, summary_df


try:
    qualitative_per_image_df, qualitative_summary_df = test_models_qualitative(
        DATASET_ROOT,
        model_names=list(models_config.keys()),
        num_samples=40,
        conf_thresholds=(0.25, 0.5, 0.7),
        iou_match_threshold=0.5
    )
except Exception as e:
    print(f"Qualitative testing failed: {e}")


In [ ]:
# ENGINEERING BENCHMARK: LATENCY, STABILITY, AND THROUGHPUT
# =============================================================================
# Benchmarks inference across multiple deployment scenarios (img sizes).
# Outputs:
# 1) Scenario-level benchmark dataframe (mean/p50/p95 latency, FPS)
# 2) Aggregated per-model report
# 3) Comparative plots for model trade-off analysis
# =============================================================================

def load_model_for_benchmark(model_name):
    best_path = Path(PROJECT_NAME) / model_name / "weights" / "best.pt"
    if best_path.exists():
        return YOLO(str(best_path)), str(best_path)
    fallback = models_config.get(model_name, "yolov8n.pt")
    return YOLO(fallback), fallback


def benchmark_model_scenario(model, img_size=640, runs=80, warmup=15):
    input_tensor = torch.randn(1, 3, img_size, img_size).to(DEVICE)

    for _ in range(warmup):
        model.predict(input_tensor, verbose=False)

    latencies = []
    for _ in range(runs):
        if DEVICE == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        model.predict(input_tensor, verbose=False)
        if DEVICE == 'cuda':
            torch.cuda.synchronize()
        t1 = time.perf_counter()
        latencies.append((t1 - t0) * 1000)

    latencies = np.array(latencies, dtype=float)
    return {
        "Latency_ms_mean": float(latencies.mean()),
        "Latency_ms_p50": float(np.percentile(latencies, 50)),
        "Latency_ms_p95": float(np.percentile(latencies, 95)),
        "Latency_ms_std": float(latencies.std()),
        "FPS": float(1000.0 / latencies.mean()) if latencies.mean() > 0 else 0.0
    }


def run_engineering_benchmark(model_names=None, scenario_img_sizes=(320, 640, 960), runs=80):
    print(f"\n{'='*10} ENGINEERING BENCHMARK {'='*10}")

    if model_names is None:
        model_names = list(benchmark_results.keys()) if benchmark_results else list(models_config.keys())

    reports_dir = Path(PROJECT_NAME) / "reports"
    figs_dir = Path(PROJECT_NAME) / "figures"
    reports_dir.mkdir(parents=True, exist_ok=True)
    figs_dir.mkdir(parents=True, exist_ok=True)

    scenario_records = []

    for model_name in model_names:
        if "NAS" in model_name:
            print(f"Skipping {model_name}: engineering benchmark currently supports Ultralytics models only.")
            continue

        try:
            model, weights_used = load_model_for_benchmark(model_name)
            print(f"Benchmarking {model_name} ({weights_used})")

            for img_size in scenario_img_sizes:
                stats = benchmark_model_scenario(model, img_size=img_size, runs=runs)
                stats.update({
                    "Model": model_name,
                    "Scenario": f"imgsz_{img_size}",
                    "Image_Size": int(img_size),
                    "Runs": int(runs),
                    "Weights": weights_used,
                    "Device": DEVICE
                })
                scenario_records.append(stats)

        except Exception as e:
            print(f"Benchmark failed for {model_name}: {e}")

    scenario_df = pd.DataFrame(scenario_records)
    if scenario_df.empty:
        print("No benchmark results produced.")
        return None, None

    model_summary_df = (
        scenario_df
        .groupby("Model", as_index=False)
        .agg(
            Mean_Latency_ms=("Latency_ms_mean", "mean"),
            Best_FPS=("FPS", "max"),
            Worst_p95_ms=("Latency_ms_p95", "max"),
            Mean_Stability_ms=("Latency_ms_std", "mean")
        )
    )

    scenario_csv = reports_dir / "engineering_benchmark_scenarios.csv"
    summary_csv = reports_dir / "engineering_benchmark_summary.csv"
    scenario_df.to_csv(scenario_csv, index=False)
    model_summary_df.to_csv(summary_csv, index=False)

    print("\n--- ENGINEERING BENCHMARK (SCENARIOS) ---")
    print(scenario_df.sort_values(["Image_Size", "Latency_ms_mean"]).to_string(index=False))
    print(f"Saved scenario benchmark to: {scenario_csv}")
    print(f"Saved summary benchmark to: {summary_csv}")

    # Backfill baseline latency/fps into benchmark_results for downstream final chart.
    baseline = scenario_df[scenario_df["Image_Size"] == IMG_SIZE]
    if not baseline.empty:
        for _, row in baseline.iterrows():
            benchmark_results.setdefault(row["Model"], {})
            benchmark_results[row["Model"]]["Latency_ms"] = float(row["Latency_ms_mean"])
            benchmark_results[row["Model"]]["FPS"] = float(row["FPS"])

    plt.style.use('seaborn-v0_8-whitegrid')

    plt.figure(figsize=(11, 6))
    sns.lineplot(data=scenario_df, x="Image_Size", y="Latency_ms_mean", hue="Model", marker="o")
    plt.title("Mean Latency vs Image Size")
    plt.xlabel("Image Size")
    plt.ylabel("Latency (ms)")
    plt.tight_layout()
    plt.savefig(figs_dir / "benchmark_latency_vs_imgsize.png", dpi=200)
    plt.show()

    plt.figure(figsize=(11, 6))
    sns.lineplot(data=scenario_df, x="Image_Size", y="FPS", hue="Model", marker="o")
    plt.title("FPS vs Image Size")
    plt.xlabel("Image Size")
    plt.ylabel("FPS")
    plt.axhline(30, color='red', linestyle='--', label='30 FPS Requirement')
    plt.legend()
    plt.tight_layout()
    plt.savefig(figs_dir / "benchmark_fps_vs_imgsize.png", dpi=200)
    plt.show()

    latency_heatmap = scenario_df.pivot(index="Model", columns="Image_Size", values="Latency_ms_mean")
    plt.figure(figsize=(8, 5))
    sns.heatmap(latency_heatmap, annot=True, fmt=".2f", cmap="mako")
    plt.title("Latency Heatmap (ms)")
    plt.tight_layout()
    plt.savefig(figs_dir / "benchmark_latency_heatmap.png", dpi=200)
    plt.show()

    p95_view = scenario_df[scenario_df["Image_Size"] == IMG_SIZE].copy()
    if not p95_view.empty:
        plt.figure(figsize=(10, 5))
        sns.barplot(data=p95_view.sort_values("Latency_ms_p95"), x="Model", y="Latency_ms_p95", palette="crest")
        plt.title(f"P95 Latency at Image Size {IMG_SIZE}")
        plt.xlabel("Model")
        plt.ylabel("P95 Latency (ms)")
        plt.xticks(rotation=20)
        plt.tight_layout()
        plt.savefig(figs_dir / "benchmark_p95_baseline.png", dpi=200)
        plt.show()

    return scenario_df, model_summary_df


try:
    benchmark_scenarios_df, benchmark_model_summary_df = run_engineering_benchmark(
        model_names=list(benchmark_results.keys()) if benchmark_results else list(models_config.keys()),
        scenario_img_sizes=(320, 640, 960),
        runs=80
    )
except Exception as e:
    print(f"Engineering benchmark failed: {e}")


In [ ]:
# VISUALIZATION & REPORT GENERATION
# =============================================================================
# Generates the 'Pareto Frontier' plot (Accuracy vs Speed).
# This is the key deliverable for the research paper.
# =============================================================================

df = pd.DataFrame(benchmark_results).T.reset_index()
df.rename(columns={'index': 'Model'}, inplace=True)

print("\n--- FINAL RESEARCH RESULTS ---")
print(df)

# Plotting
plt.figure(figsize=(12, 8))
sns.set_style("whitegrid")

# Bubble Chart: X=Latency, Y=mAP, Size=Params
scatter = sns.scatterplot(
    data=df, 
    x='Latency_ms', 
    y='mAP50', 
    hue='Model', 
    size='Parameters',
    sizes=(100, 1000),
    alpha=0.7,
    palette='viridis'
)

# Labeling
for line in range(0, df.shape[0]):
    plt.text(
        df.Latency_ms[line]+0.2, 
        df.mAP50[line], 
        df.Model[line], 
        horizontalalignment='left', 
        size='medium', 
        color='black', 
        weight='semibold'
    )

plt.title("Pareto Frontier: Pothole Detection (Accuracy vs. Edge Latency)", fontsize=16)
plt.xlabel("Inference Latency (ms) [Lower is Better]", fontsize=12)
plt.ylabel("mAP @ 50 [Higher is Better]", fontsize=12)
plt.grid(True, which='both', linestyle='--', linewidth=0.5)

# Add Drone limit line (e.g., 30 FPS = ~33ms)
plt.axvline(x=33.3, color='r', linestyle='--', label='Min 30 FPS Requirement')
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)

plt.tight_layout()
plt.savefig(f"{PROJECT_NAME}/benchmark_results.png")
plt.show()

print(f"Analysis Complete. Results saved to {PROJECT_NAME}/benchmark_results.png")